# Train and evaluate Responsible Credit AI

This notebook teaches the workflow and trains the same model bundle used by the website. Run cells from top to bottom. The model search may take a few minutes.

> **Important:** a website prediction does not train the model. Training requires examples with known correct labels. The German Credit data has labels; demo applicants do not. Never treat an unverified review or prediction as a training label.

The training code keeps a 20% test split untouched while comparing algorithms and feature sets with cross-validation on the other 80%. It then chooses a threshold from training-only out-of-fold predictions and evaluates the selected model on the held-out test data.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'app').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)

## 1. Inspect the labelled dataset
Each row is one historical case. `credit_risk` is the answer used for training: 0 = good risk, 1 = bad risk.

In [ ]:
from app.dataset import load_dataset, CATEGORICAL_FEATURES, NUMERIC_FEATURES
X, y = load_dataset()
print(f'{len(X)} rows × {X.shape[1]} input fields')
print(f'{len(CATEGORICAL_FEATURES)} categorical, {len(NUMERIC_FEATURES)} numeric/ordinal fields')
display(pd.DataFrame({'label': ['good risk (0)', 'bad risk (1)'], 'count': [int((y == 0).sum()), int((y == 1).sum())]}))
display(X.head())

## 2. Train and test the model candidates
This calls the project's actual training pipeline. Preprocessing is fitted inside each cross-validation fold. The held-out test split is used only for the final evaluation.

In [ ]:
from app.training import main as train_and_save
train_and_save()
print('Training finished. The website model and report have been refreshed.')

## 3. Review the cross-validation comparison
Accuracy is the share classified correctly. Bad-risk recall is the share of bad-risk cases found. The cross-validation values are training-only estimates, not final test results.

In [ ]:
report_path = PROJECT_ROOT / 'reports' / 'model_report.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
cv = pd.DataFrame(report['cv_comparison'])
display(cv[['name', 'cv_accuracy_mean', 'cv_accuracy_std', 'cv_balanced_accuracy_mean', 'cv_bad_risk_recall_mean', 'cv_expected_error_cost_mean']].style.format(precision=3))
print('Selected:', report['model_name'])
print('Features:', len(report['selected_features']), '| Threshold:', round(report['threshold'], 2))

## 4. Understand held-out test results
The two rows use the same final model probabilities with different decision thresholds. A lower threshold catches more bad-risk cases here, but can also label more good-risk cases as bad risk.

In [ ]:
test_rows = []
for label, key in [('Standard threshold (0.50)', 'test_metrics_at_0_5'), ('Cost-aware threshold', 'test_metrics')]:
    m = report[key]
    test_rows.append({'operating point': label, 'threshold': m['threshold'], 'accuracy': m['accuracy'], 'balanced accuracy': m['balanced_accuracy'], 'bad-risk precision': m['bad_risk_precision'], 'bad-risk recall': m['bad_risk_recall'], 'ROC AUC': m['roc_auc'], 'cost per case': m['expected_error_cost_per_case']})
display(pd.DataFrame(test_rows).set_index('operating point').style.format(precision=3))
print('Majority-class accuracy baseline:', f"{report['test_metrics']['majority_baseline_accuracy']:.3f}")
print('These are estimates from only', report['test_metrics']['test_rows'], 'held-out records.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, title, key in zip(axes, ['Threshold 0.50', 'Cost-aware threshold'], ['test_metrics_at_0_5', 'test_metrics']):
    cm = report[key]['confusion_matrix_good_then_bad']
    ax.imshow(cm, cmap='Greens')
    ax.set_title(title)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('Actual label')
    ax.set_xticks([0, 1], ['Good', 'Bad'])
    ax.set_yticks([0, 1], ['Good', 'Bad'])
    for (r, c), value in __import__('numpy').ndenumerate(cm):
        ax.text(c, r, str(value), ha='center', va='center')
plt.tight_layout()
plt.show()

## 5. Try the saved model on one example
This only predicts; it does not update model weights. The example comes from the dataset and is for learning, not a real applicant.

In [ ]:
import joblib
bundle = joblib.load(PROJECT_ROOT / 'app' / 'model.joblib')
sample = X.iloc[[0]][bundle['feature_names']]
p_bad = float(bundle['pipeline'].predict_proba(sample)[0, 1])
print('Estimated bad-risk probability:', f'{p_bad:.1%}')
print('Predicted dataset label:', 'bad risk' if p_bad >= bundle['threshold'] else 'good risk')
print('Model version:', bundle['model_version'])

## What to remember
- Training learns from labelled examples; testing checks predictions against known labels.
- Cross-validation compares candidates while keeping the final test split aside.
- A website prediction is not a label, so it cannot teach the model on its own.
- To learn from new cases, first collect verified outcomes, then version the labelled dataset and rerun this notebook. Do not use guessed or model-generated labels.